# K05 demo — concurrency with `asyncio.gather`

Run this top-to-bottom (**Kernel → Restart Kernel and Run All Cells**) and compare the two
timings. The concurrent version does the *same work* in about half the wall-clock time.

In [ ]:
# Setup cell: imports only.
import asyncio
import time

In [ ]:
async def fetch(name: str, seconds: float) -> str:
    """Pretend to do slow I/O (stand-in for an LLM call or a tool call)."""
    await asyncio.sleep(seconds)
    return f"{name} done"

## Sequential — one `await` after another

In [ ]:
start = time.perf_counter()
r1 = await fetch("A", 1.0)
r2 = await fetch("B", 1.0)
print(r1, r2)
print(f"sequential took {time.perf_counter() - start:.2f}s")

## Concurrent — `asyncio.gather` runs them together

In [ ]:
start = time.perf_counter()
results = await asyncio.gather(fetch("A", 1.0), fetch("B", 1.0))
print(results)
print(f"gather took {time.perf_counter() - start:.2f}s")

## Why this matters

Sequential took ~2s; `gather` took ~1s — **same work, half the wall-clock**, because the two
`await`s overlap instead of blocking each other. This is why agent code is written async:
concurrent tool calls, running multiple agents / eval cases at once, and non-blocking traces
all fall out of `await` + `asyncio.gather`. See the K05 guide's *"why async for agents."*